In [2]:
# Install (run once — you can comment this out after first run)
# !pip install duckdb scikit-learn imbalanced-learn

In [3]:
# Imports
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, ConfusionMatrixDisplay


In [31]:
# Filter to resolved loans only and select relevant columns
df = duckdb.query("""
    SELECT loan_amnt,
           funded_amnt,
           term,
           int_rate,
           grade,
           sub_grade,
           emp_length,
           home_ownership,
           annual_inc,
           verification_status,
           loan_status,
           dti,
           delinq_2yrs,
           earliest_cr_line,
           inq_last_6mths,
           mths_since_last_delinq,
           open_acc,
           pub_rec,
           revol_bal,
           revol_util,
           total_acc,
           issue_d,
           addr_state,
           purpose,
           pub_rec_bankruptcies,
           mort_acc,
           num_accts_ever_120_pd,
           bc_util,
           pct_tl_nvr_dlq
    FROM 'lending_club.csv'
    WHERE loan_status IN ('Charged Off', 'Fully Paid')
""").df()

print(df.shape)
print(df.memory_usage(deep=True).sum() / 1e6, "MB")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(1303607, 29)


1002.925061 MB


In [34]:
df.head(25)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1303607 entries, 0 to 1303606
Data columns (total 29 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   loan_amnt               1303607 non-null  int32  
 1   funded_amnt             1303607 non-null  int64  
 2   term                    1303607 non-null  str    
 3   int_rate                1303607 non-null  float64
 4   grade                   1303607 non-null  str    
 5   sub_grade               1303607 non-null  str    
 6   emp_length              1303607 non-null  str    
 7   home_ownership          1303607 non-null  str    
 8   annual_inc              1303607 non-null  float64
 9   verification_status     1303607 non-null  str    
 10  loan_status             1303607 non-null  str    
 11  dti                     1303295 non-null  float64
 12  delinq_2yrs             1303607 non-null  int64  
 13  earliest_cr_line        1303607 non-null  str    
 14  inq_last_6mth

In [35]:
# Check max values on all integer columns before deciding dtype
int_cols = ['loan_amnt', 'funded_amnt', 'delinq_2yrs', 'inq_last_6mths', 'mths_since_last_delinq',
            'open_acc', 'pub_rec', 'revol_bal', 'total_acc', 'pub_rec_bankruptcies', 'mort_acc',
            'num_accts_ever_120_pd']

df[int_cols].agg(['min', 'max']).T

,min,max
loan_amnt,500,40000
funded_amnt,500,40000
delinq_2yrs,0,39
inq_last_6mths,0,8
mths_since_last_delinq,0,226
open_acc,0,90
pub_rec,0,86
revol_bal,0,2904836
total_acc,2,176
pub_rec_bankruptcies,0,12


## All int64 columns — downcast to so smaller int# dtypes:

### Use Int# instead of int#.  Capital 'I' preserves NaN values

In [38]:
# int8: -128 to 127

int8_cols = ['delinq_2yrs','inq_last_6mths','open_acc','pub_rec','pub_rec_bankruptcies',
             'mort_acc','num_accts_ever_120_pd']

for col in int8_cols:
    df[col] = df[col].astype('Int8')   


In [40]:
# int16: -32,768 to 32767

int16_cols = ['mths_since_last_delinq','mths_since_last_delinq']

for col in int16_cols:
    df[col] = df[col].astype('Int16')

In [43]:
# int32: up to ~2.1 billion

int32_cols = ['loan_amnt','funded_amnt','revol_bal','total_acc']

for col in int32_cols:
    df[col] = df[col].astype('Int32')

### Reduce float64 columns same as the int64 columns

In [46]:
float_cols = ['int_rate', 'annual_inc', 'dti', 'revol_util', 'bc_util' ,'pct_tl_nvr_dlq']

df[float_cols].agg(['min', 'max']).T

,min,max
int_rate,5.31,30.99
annual_inc,0.00,10999200.00
dti,-1.00,999.00
revol_util,0.00,892.30
bc_util,0.00,339.60
pct_tl_nvr_dlq,0.00,100.00


In [48]:
# float32: +/- 3.4 x 10**38, precision: 7 decimal digits

# float64: +/- 1.8 x 10**308, precision: 15 decimal digits

float32_cols = ['int_rate', 'annual_inc', 'dti', 'revol_util', 'bc_util' ,'pct_tl_nvr_dlq']

for col in float32_cols:
    df[col] = df[col].astype('Float32')

In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1303607 entries, 0 to 1303606
Data columns (total 29 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   loan_amnt               1303607 non-null  Int32  
 1   funded_amnt             1303607 non-null  Int32  
 2   term                    1303607 non-null  str    
 3   int_rate                1303607 non-null  Float32
 4   grade                   1303607 non-null  str    
 5   sub_grade               1303607 non-null  str    
 6   emp_length              1303607 non-null  str    
 7   home_ownership          1303607 non-null  str    
 8   annual_inc              1303607 non-null  Float32
 9   verification_status     1303607 non-null  str    
 10  loan_status             1303607 non-null  str    
 11  dti                     1303295 non-null  Float32
 12  delinq_2yrs             1303607 non-null  Int8   
 13  earliest_cr_line        1303607 non-null  str    
 14  inq_last_6mth

In [50]:
print(df.memory_usage(deep=True).sum() / 1e6, "MB")

896.029287 MB


### Converting str columns with few values to dtype category

#### Review claude suggested columns ['term', 'grade', 'sub_grade', 'emp_length', 'home_ownership',
####                                  'verification_status', 'loan_status', 'purpose', 'addr_state']

In [52]:
# term should be fixed to just Int8; remove months and extra spaces
df['term'].unique()
df['term'] = df['term'].str.replace('months', '').str.strip().astype('Int8')

<StringArray>
[' 36 months', ' 60 months']
Length: 2, dtype: str

In [6]:
df['addr_state'].unique()            # ok as category

df['grade'].unique()                 # ok as category
df['sub_grade'].unique()             # ok as category
df['emp_length'].unique()            # ok as category
df['home_ownership'].unique()        # ok as category
df['verification_status'].unique()   # ok as category
df['loan_status'].unique()           # ok as category
df['purpose'].unique()               # ok as category


NameError: name 'df' is not defined

In [67]:
cat_cols = ['grade', 'sub_grade', 'emp_length', 'home_ownership', 'verification_status', 
            'loan_status', 'purpose', 'addr_state', '', '']

for col in cat_cols:
    df[col] = df[col].astype('category')

In [69]:
print(df.memory_usage(deep=True).sum() / 1e6, "MB")
df.info()

248.995924 MB


<class 'pandas.DataFrame'>
RangeIndex: 1303607 entries, 0 to 1303606
Data columns (total 29 columns):
 #   Column                  Non-Null Count    Dtype   
---  ------                  --------------    -----   
 0   loan_amnt               1303607 non-null  Int32   
 1   funded_amnt             1303607 non-null  Int32   
 2   term                    1303607 non-null  Int8    
 3   int_rate                1303607 non-null  Float32 
 4   grade                   1303607 non-null  category
 5   sub_grade               1303607 non-null  category
 6   emp_length              1303607 non-null  category
 7   home_ownership          1303607 non-null  category
 8   annual_inc              1303607 non-null  Float32 
 9   verification_status     1303607 non-null  category
 10  loan_status             1303607 non-null  category
 11  dti                     1303295 non-null  Float32 
 12  delinq_2yrs             1303607 non-null  Int8    
 13  earliest_cr_line        1303607 non-null  str     
 1

In [72]:
df['addr_state'].nunique()

51

In [75]:
df['default'] = (df['loan_status'] == 'Charged Off').astype('Int8')

In [83]:
# Save to parquet
# df.to_parquet('lending_club_clean.parquet', index=False) 
# bombs out

# df.to_parquet('lending_club_clean.parquet', index=False, engine='pyarrow') 
# ArrowKeyError: A type extension with name pandas.period already defined


In [5]:
# df_save = df.copy()
# df_save = df_save.convert_dtypes(dtype_backend='numpy_nullable')
# df_save.to_parquet('lending_club_clean.parquet', index=False)

# print("Saved successfully")

In [85]:
!pip install fastparquet

   ---------------------------------------- 0.0/695.2 kB ? eta -:--:--
   ---------------------------------------- 695.2/695.2 kB 8.7 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 14.0 MB/s  0:00:00

   -------------------- ------------------- 1/2 [fastparquet]
   ---------------------------------------- 2/2 [fastparquet]



In [86]:
df.to_parquet('lending_club_clean.parquet', index=False, engine='fastparquet')

In [87]:
# Clean summary of dtype and unique values for every column
summary = pd.DataFrame({
    'dtype': df.dtypes,
    'unique_values': df.nunique()
})
print(summary)

                           dtype  unique_values
loan_amnt                  Int32           1553
funded_amnt                Int32           1553
term                        Int8              2
int_rate                 Float32            654
grade                   category              7
sub_grade               category             35
emp_length              category             12
home_ownership          category              6
annual_inc               Float32          62988
verification_status     category              3
loan_status             category              2
dti                      Float32           6870
delinq_2yrs                 Int8             30
earliest_cr_line             str            738
inq_last_6mths              Int8              9
mths_since_last_delinq     Int16            163
open_acc                    Int8             84
pub_rec                     Int8             37
revol_bal                  Int32          82819
revol_util               Float32        

In [88]:
df['earliest_cr_line'] = df['earliest_cr_line'].astype('category')
df['issue_d'] = df['issue_d'].astype('category')

print(df.memory_usage(deep=True).sum() / 1e6, "MB")

108.256465 MB


In [89]:
df.to_parquet('lending_club_clean.parquet', index=False, engine='fastparquet')
print("Saved - final size:", df.memory_usage(deep=True).sum() / 1e6, "MB")

Saved - final size: 108.256465 MB


In [3]:
df.info()

NameError: name 'df' is not defined